# PARC2026 — 72b SmolVLA A100 batch probe

SmolVLA固定commitを専用Python 3.12環境で1 optimizer stepだけ動かし、effective batch 32を保ったまま最大micro-batchを探索します。probe-onlyで、M3 benchmark trainingではありません。

pretrained `lerobot/smolvla_base` の重みは維持しつつ、`input_features/output_features=null` でselected LIBERO datasetの `front/wrist + state8 + action7` schemaを再推論します。失敗時はDriveのresult/logを自動表示します。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')

ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_probe'
PIN = '5c334095e327c56173e06eed4c5f19fae3fc7878'
URL = 'https://github.com/yu37330/py_AI.git'

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('72b probe code:', got, flush=True)

env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)

try:
    subprocess.run(
        ['python', '-u', str(REPO / 'tools/colab/run_m3_smolvla_batch_probe.py')],
        cwd=str(REPO),
        env=env,
        check=True,
    )
except subprocess.CalledProcessError:
    print('\n=== 72b DIAGNOSTICS ===', flush=True)
    DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
    result_path = DRIVE / 'model-benchmark-v1/batch-probes/smolvla.json'
    if result_path.is_file():
        print('\n--- SMOLVLA RESULT ---', flush=True)
        print(result_path.read_text(encoding='utf-8', errors='replace'), flush=True)

    log_root = DRIVE / 'model-benchmark-v1/batch-probes/logs/smolvla'
    candidates = sorted(
        log_root.glob('*.log'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    ) if log_root.exists() else []
    if candidates:
        latest = candidates[0]
        print('\n--- LATEST SMOLVLA LOG ---', flush=True)
        print('Log:', latest, flush=True)
        lines = latest.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-250:]), flush=True)
    else:
        print('\nNo SmolVLA candidate log found; failure happened before lerobot-train started.', flush=True)
    raise

print('=== 72b COMPLETE ===', flush=True)
print('Probe only. M3 benchmark training has NOT started.', flush=True)
